# Weather Forecasting Tutorial

## Step-by-Step Guide to Using the Weather Library

This notebook provides a complete walkthrough of the LCCC Weather library for renewable energy forecasting using ERA5 weather data and Monte Carlo sampling techniques.

**Learning Outcomes:**
- Install and setup the weather library
- Load historical weather data
- Generate Monte Carlo weather samples
- Visualize forecast results and uncertainties
- Analyze statistical properties of generated samples

## Step 1: Installation and Setup

First, ensure the weather library is installed with all necessary dependencies.

In [ ]:
# Installation instructions (run in terminal, not in notebook):
# uv sync --group notebook  # Install with notebook dependencies (Jupyter, Plotly, etc.)
# OR
# pip install weather-lccc jupyter ipykernel plotly

print("Weather library is ready to use!")

## Step 2: Import Required Libraries

Load the main weather library modules along with data science and visualization tools.

In [ ]:
import datetime
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from weather.core.data_loader import LocalDataLoader
from weather.simulation.weather_data import HistoricalMetadata, WeatherData

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")

## Step 3: Initialize Data Loader

The `LocalDataLoader` is the entry point for accessing historical weather and plant data.

In [ ]:
# Initialize the data loader
loader = LocalDataLoader()

print("✓ Data loader initialized")
print(f"  Data loader type: {type(loader).__name__}")
print(f"  Ready to load weather, plant, and generation data")

## Step 4: Load Historical Weather Metadata

Check what historical weather data is available in your configured data path.

In [ ]:
# Get the manifest of available weather data
manifest_wind = loader.check_historical_weather()

print("✓ Historical weather metadata loaded")
print(f"\nManifest Contents:")
for key, value in manifest_wind.items():
    if key != 'artifact' and key != 'artifact_histogram':
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: [data object]")

## Step 5: Create HistoricalMetadata Object

Convert the manifest into a `HistoricalMetadata` object that the sampler can use.

In [ ]:
# Create metadata object from manifest
metadata_wind = HistoricalMetadata(
    manifest_wind["basename"],
    loader.path_resolver_weather_data,
    datetime.datetime.fromisoformat(manifest_wind["horizon_utc"]["start"]),
    datetime.datetime.fromisoformat(manifest_wind["horizon_utc"]["end"]),
    manifest_wind["artifact"]["columns"],
    manifest_wind["artifact_histogram"]["rows_per_block"],
)

print("✓ HistoricalMetadata object created")
print(f"\nMetadata Details:")
print(f"  Basename: {metadata_wind.basename}")
print(f"  Start Date: {metadata_wind.start}")
print(f"  End Date: {metadata_wind.end}")
print(f"  Number of Columns: {len(metadata_wind.columns)}")
print(f"  Rows Per Block: {metadata_wind.rows_per_block}")

## Step 6: Load Supporting Data

Load prefix histograms and historical weather data needed for advanced sampling features.

In [ ]:
# Load prefix histograms (memory-mapped for efficiency)
prefix_histograms_wind = loader.get_prefix_histograms()

# Load historical data (optional, used for desired averages computation)
historical_data_wind = loader.get_historical_weather()

print("✓ Supporting data loaded")
print(f"  Prefix histograms type: {type(prefix_histograms_wind).__name__}")
print(f"  Historical data type: {type(historical_data_wind).__name__}")

## Step 7: Initialize Weather Data Sampler

Create a `WeatherData` object configured with metadata and optional constraints.

In [ ]:
# Initialize the weather sampler
# desired_averages: {time_step: [target_avg_1, target_avg_2, ...]}
# This constrains the sampler to match specified average values
wind_sampler = WeatherData(
    metadata=metadata_wind,
    desired_averages={1: [0.14, 0.5, 0.66]},  # Target averages for column 1
    prefix_histograms=prefix_histograms_wind,
    historical_data=historical_data_wind,
)

print("✓ Weather sampler initialized")
print(f"  Sampler type: {type(wind_sampler).__name__}")
print(f"  Desired averages configured: {wind_sampler.desired_averages}")

## Step 8: Generate a Single Weather Sample

Create one Monte Carlo sample for a future forecast period.

In [ ]:
# Define forecast period
start_date = datetime.datetime(2025, 4, 1)
end_date = datetime.datetime(2025, 8, 7)

# Generate a single sample with fixed random seeds for reproducibility
sample_1 = wind_sampler.random_sample(
    start_date,
    end_date,
    random.Random(42),           # Python random seed
    np.random.default_rng(42),   # NumPy random seed
)

print("✓ Single weather sample generated")
print(f"\nSample Details:")
print(f"  Forecast Period: {start_date.date()} to {end_date.date()}")
print(f"  Duration: {(end_date - start_date).days} days")
print(f"  Sample type: {type(sample_1).__name__}")

## Step 9: Extract and Display Sample Data

Access specific columns from the generated sample and view the data.

In [ ]:
# Extract column 1 (wind speed) from the sample
wind_speed_data = sample_1.access_col(1, 1)  # (column_id, timestep)

print("✓ Data extracted from sample")
print(f"\nWind Speed Data:")
print(f"  Length: {len(wind_speed_data)} time steps")
print(f"  Data type: {type(wind_speed_data).__name__}")
print(f"\n  First 10 values:")
print(f"  {wind_speed_data[:10]}")
print(f"\n  Basic Statistics:")
print(f"    Min: {np.min(wind_speed_data):.4f}")
print(f"    Max: {np.max(wind_speed_data):.4f}")
print(f"    Mean: {np.mean(wind_speed_data):.4f}")
print(f"    Std Dev: {np.std(wind_speed_data):.4f}")

## Step 10: Visualize Time Series

Plot the complete wind speed forecast for the selected period.

In [ ]:
# Create time index for the forecast period
time_index = pd.date_range(start=start_date, end=end_date, periods=len(wind_speed_data))

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot time series
ax.plot(time_index, wind_speed_data, linewidth=2, color='steelblue', label='Wind Speed')
ax.fill_between(time_index, wind_speed_data, alpha=0.3, color='steelblue')

# Formatting
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
ax.set_title('Wind Speed Forecast Time Series (Single Monte Carlo Sample)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"✓ Time series visualization complete")

## Step 11: Statistical Summary

Compute and display detailed statistics of the wind speed sample.

In [ ]:
# Calculate comprehensive statistics
stats_dict = {
    'Metric': ['Count', 'Mean', 'Std Dev', 'Min', '25%', 'Median', '75%', 'Max', 'Skewness', 'Kurtosis'],
    'Value': [
        len(wind_speed_data),
        np.mean(wind_speed_data),
        np.std(wind_speed_data),
        np.min(wind_speed_data),
        np.percentile(wind_speed_data, 25),
        np.median(wind_speed_data),
        np.percentile(wind_speed_data, 75),
        np.max(wind_speed_data),
        float(pd.Series(wind_speed_data).skew()),
        float(pd.Series(wind_speed_data).kurtosis())
    ]
}

stats_df = pd.DataFrame(stats_dict)
print("\n✓ Statistical Summary:")
print(stats_df.to_string(index=False))

## Step 12: Distribution Analysis

Analyze the distribution of wind speeds through histograms and box plots.

In [ ]:
# Create subplots for distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(wind_speed_data, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(wind_speed_data), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(wind_speed_data):.2f}')
axes[0].set_xlabel('Wind Speed (m/s)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Wind Speed Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
bp = axes[1].boxplot(wind_speed_data, vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('steelblue')
axes[1].set_ylabel('Wind Speed (m/s)', fontsize=11, fontweight='bold')
axes[1].set_title('Wind Speed Distribution (Box Plot)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"✓ Distribution analysis complete")

## Step 13: Generate Multiple Monte Carlo Samples

Create an ensemble of weather samples to represent uncertainty in forecasts.

In [ ]:
# Generate multiple samples with different random seeds
num_samples = 10
ensemble_samples = []

for i in range(num_samples):
    sample = wind_sampler.random_sample(
        start_date,
        end_date,
        random.Random(100 + i),           # Different seed for each sample
        np.random.default_rng(100 + i),
    )
    ensemble_samples.append(sample)
    print(f"  Generated sample {i+1}/{num_samples}")

print(f"\n✓ Ensemble of {num_samples} samples generated")

## Step 14: Ensemble Visualization with Uncertainty Bands

Plot all ensemble members with mean and percentile confidence bounds.

In [ ]:
# Extract wind speed data from all samples
ensemble_data = np.array([
    sample.access_col(1, 1) for sample in ensemble_samples
])

# Calculate statistics
ensemble_mean = np.mean(ensemble_data, axis=0)
ensemble_p10 = np.percentile(ensemble_data, 10, axis=0)
ensemble_p90 = np.percentile(ensemble_data, 90, axis=0)
ensemble_p25 = np.percentile(ensemble_data, 25, axis=0)
ensemble_p75 = np.percentile(ensemble_data, 75, axis=0)

# Create visualization
fig, ax = plt.subplots(figsize=(14, 7))

# Plot individual ensemble members (lightly)
for i, data in enumerate(ensemble_data):
    ax.plot(time_index, data, linewidth=0.8, alpha=0.3, color='gray', label='Sample' if i == 0 else '')

# Plot percentile bands
ax.fill_between(time_index, ensemble_p10, ensemble_p90, alpha=0.3, color='orange', label='P10-P90 (80% confidence)')
ax.fill_between(time_index, ensemble_p25, ensemble_p75, alpha=0.4, color='coral', label='P25-P75 (50% confidence)')

# Plot mean
ax.plot(time_index, ensemble_mean, linewidth=2.5, color='darkred', label='Ensemble Mean', zorder=10)

# Formatting
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
ax.set_title(f'Wind Speed Forecast Ensemble ({num_samples} Monte Carlo Samples)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"✓ Ensemble visualization complete with uncertainty bands")

## Step 15: Ensemble Statistics and Uncertainty Quantification

Analyze the spread and variability across the ensemble.

In [ ]:
# Calculate ensemble-level statistics
ensemble_stats = {
    'Metric': [
        'Number of Samples',
        'Mean (across ensemble)',
        'Std Dev (across ensemble)',
        'Min (all samples)',
        'Max (all samples)',
        'P10 (10th percentile)',
        'P50 (median)',
        'P90 (90th percentile)',
        'Interquartile Range (P75-P25)'
    ],
    'Value': [
        num_samples,
        f"{np.mean(ensemble_mean):.4f}",
        f"{np.std(ensemble_mean):.4f}",
        f"{np.min(ensemble_data):.4f}",
        f"{np.max(ensemble_data):.4f}",
        f"{np.percentile(ensemble_data, 10):.4f}",
        f"{np.percentile(ensemble_data, 50):.4f}",
        f"{np.percentile(ensemble_data, 90):.4f}",
        f"{np.mean(ensemble_p75 - ensemble_p25):.4f}"
    ]
}

ensemble_stats_df = pd.DataFrame(ensemble_stats)
print("\n✓ Ensemble Statistical Summary:")
print(ensemble_stats_df.to_string(index=False))

# Uncertainty over time
time_uncertainty = np.std(ensemble_data, axis=0)
print(f"\n  Average Uncertainty (Std Dev): {np.mean(time_uncertainty):.4f}")
print(f"  Max Uncertainty: {np.max(time_uncertainty):.4f}")
print(f"  Min Uncertainty: {np.min(time_uncertainty):.4f}")

## Step 16: Plant Data Integration (Optional)

Load and view plant configuration data for wind and solar installations.

In [ ]:
# Load plant data (optional)
try:
    plant_ds = loader.load_plant_data()
    wind_plants = plant_ds.get_wind_plants()
    print("✓ Plant data loaded successfully")
    print(f"\n  Wind plants available: {wind_plants}")
except Exception as e:
    print(f"ℹ Plant data not available: {str(e)}")
    print("  (This is normal if plant data hasn't been configured)")

## Step 17: Key Takeaways and Next Steps

Summary of what you've learned and how to extend this tutorial.

In [ ]:
print("\n" + "="*60)
print("KEY TAKEAWAYS")
print("="*60)

print("\n1. LIBRARY COMPONENTS:")
print("   - LocalDataLoader: Access to weather, plant, and generation data")
print("   - HistoricalMetadata: Metadata about available historical data")
print("   - WeatherData: Monte Carlo sampler for future forecasts")

print("\n2. WORKFLOW:")
print("   Step 1: Initialize LocalDataLoader")
print("   Step 2: Load historical weather manifest")
print("   Step 3: Create HistoricalMetadata object")
print("   Step 4: Initialize WeatherData sampler")
print("   Step 5: Generate samples for future periods")
print("   Step 6: Extract and analyze results")

print("\n3. MONTE CARLO SAMPLING:")
print("   - Multiple samples represent uncertainty")
print("   - Percentile bands show confidence intervals")
print("   - Ensemble mean provides best estimate")

print("\n4. CUSTOMIZATION OPTIONS:")
print("   - Desired averages: Constrain forecast to match targets")
print("   - Random seeds: Control reproducibility")
print("   - Ensemble size: Balance accuracy vs. computation")

print("\n5. NEXT STEPS:")
print("   - Load plant configuration data")
print("   - Convert wind speed to power output")
print("   - Analyze financial implications")
print("   - Integrate with grid planning tools")
print("   - Implement custom calibration workflows")

print("\n" + "="*60)
print("For more information, see README.md and documentation")
print("="*60)

## Appendix: Common Use Cases

### Generate Forecasts for Different Periods

In [ ]:
# Example: Generate forecasts for different future periods
# future_periods = [
#     (datetime.datetime(2025, 6, 1), datetime.datetime(2025, 6, 30)),  # June 2025
#     (datetime.datetime(2025, 12, 1), datetime.datetime(2025, 12, 31)),  # December 2025 (winter)
# ]
#
# for start, end in future_periods:
#     sample = wind_sampler.random_sample(
#         start, end,
#         random.Random(999),
#         np.random.default_rng(999)
#     )
#     data = sample.access_col(1, 1)
#     print(f"Period {start.date()} to {end.date()}: Mean={np.mean(data):.4f}, Std={np.std(data):.4f}")

print("See commented code above for examples of generating multiple forecast periods")

### Seasonal Analysis

In [ ]:
# Example: Compare seasonal variations
# seasons = {
#     'Spring': (datetime.datetime(2025, 3, 21), datetime.datetime(2025, 6, 20)),
#     'Summer': (datetime.datetime(2025, 6, 21), datetime.datetime(2025, 9, 22)),
#     'Fall': (datetime.datetime(2025, 9, 23), datetime.datetime(2025, 12, 21)),
#     'Winter': (datetime.datetime(2025, 12, 22), datetime.datetime(2026, 3, 20))
# }
#
# seasonal_stats = {}
# for season, (start, end) in seasons.items():
#     sample = wind_sampler.random_sample(
#         start, min(end, datetime.datetime(2025, 12, 31)),
#         random.Random(999),
#         np.random.default_rng(999)
#     )
#     data = sample.access_col(1, 1)
#     seasonal_stats[season] = {"mean": np.mean(data), "std": np.std(data)}
#
# seasonal_df = pd.DataFrame(seasonal_stats).T
# print(seasonal_df)

print("See commented code above for seasonal analysis examples")